# Practice Lab: Remote MCP Servers on Cloud Run (Lesson 10.6)

Module 10 · 8 exercises · Dockerfile + Cloud Run + autoscaling + CI/CD + OTel + OAuth + India

Exercises 1, 3, 5 run locally (pure Python). Rest are design.


# Lesson 10.6: Remote MCP Servers on Cloud Run

8 exercises: 2E + 3M + 3C

Exercises 1, 3, 5 run **locally** (pure Python simulations). Rest are design.


In [ ]:
import json
import math


---
## Exercise 1 (Easy): Optimized Dockerfile


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Dockerfile Optimization:")
print("=" * 55)

print(f"\n  {'Metric':<20} {'Naive':>12} {'Optimized':>12} {'Savings':>8}")
print(f"  {'-'*54}")
for name,n,o,s in [("Image size","1020MB","148MB","85%"),("Cold start","6.5s","1.8s","72%"),
    ("Install time","45s","4s","91%"),("User","root","appuser","security"),("Bytecode","runtime","pre-compiled","faster")]:
    print(f"  {name:<20} {n:>12} {o:>12} {s:>8}")

print(f"\nOptimized Dockerfile:")
for line in ["FROM python:3.13-slim","RUN useradd -m appuser","WORKDIR /app",
    "COPY --from=ghcr.io/astral-sh/uv /uv /bin/uv","ENV UV_COMPILE_BYTECODE=1",
    "COPY pyproject.toml uv.lock ./","RUN uv sync --frozen --no-dev",
    "COPY server.py .","USER appuser","ENV PORT=8080","CMD ['uv', 'run', 'server.py']"]:
    print(f"  {line}")

print(f"\n5 Rules: slim not full, uv not pip, non-root, pre-compile bytecode, deps-first layers")


</details>


---
## Exercise 2 (Easy): Cloud Run Deployment Design


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Cloud Run MCP Deployment:")
reqs = 10000 * 30; avg_ms = 400; vcpu_s = reqs * avg_ms / 1000
cost = vcpu_s * 0.00002400 + vcpu_s * 0.00000250 * 256/1024
print(f"  Deploy: gcloud run deploy netsetos-mcp --source . --region asia-south1")
print(f"  Flags: --memory 256Mi --min-instances 0 --max-instances 10 --cpu-boost")
print(f"  Transport: StreamableHTTPServerTransport(endpoint='/mcp')")
print(f"  Health: @starlette_app.route('/') -> {{'status':'healthy'}}")
print(f"\n  Stateless vs Stateful:")
print(f"    Stateless: horizontal scaling, no sampling/elicitation -> DEFAULT")
print(f"    Stateful: sticky sessions, sampling works -> single-instance")
print(f"\n  Cost (10K req/day, Mumbai): ~${cost:.2f}/month + $3.24 warm instance")
print(f"  Mumbai: NO free tier (15-20% premium)")


</details>


---
## Exercise 3 (Medium): Autoscaling Tuning


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math

print("Autoscaling Tuning:")
print(f"\n  Instances Needed (60% target):")
print(f"  {'Users':<10} {'Conc=80':>10} {'Conc=20':>10} {'Conc=10':>10}")
print(f"  {'':10} {'(I/O)':>10} {'(mixed)':>10} {'(CPU)':>10}")
for u in [100, 500, 1000]:
    io = math.ceil(u/(80*0.6)); mx = math.ceil(u/(20*0.6)); cp = math.ceil(u/(10*0.6))
    print(f"  {u:<10} {io:>10} {mx:>10} {cp:>10}")

print(f"\n  Cold Start Mitigation:")
for s,e,c in [("min-instances=1","No 0->1 cold start","$3.24/mo"),("cpu-boost","2x CPU at startup","free"),
    ("slim image","150MB vs 1GB","3-5x faster"),("UV_COMPILE_BYTECODE","Pre-compiled .pyc","~200ms saved"),
    ("Lazy imports","Heavy libs in function","variable")]:
    print(f"    {s:<22} {e:<28} {c}")

print(f"\n  Gen2: MicroVMs, better CPU, 512MiB min. Use + min-instances=1")
print(f"  min-instances cost: 0=$0, 1=$3.24, 2=$6.48/month")


</details>


---
## Exercise 4 (Medium): GitHub Actions CI/CD Design


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("GitHub Actions CI/CD:")
for name,desc,t in [("Test","pytest + Client(server)","~10s"),("Lint","ruff + mypy","~5s"),
    ("Build","Docker + Artifact Registry","~60s"),("Deploy","--no-traffic (canary)","~30s"),
    ("Validate","Inspector --cli against URL","~15s"),("Promote","10% -> 50% -> 100%","~5s")]:
    print(f"  {name:<10} {desc:<40} {t}")
print(f"\n  Auth: Workload Identity Federation (no JSON keys)")
print(f"    permissions: id-token: write")
print(f"    uses: google-github-actions/auth@v2")
print(f"  Canary: deploy --no-traffic -> update-traffic LATEST=10 -> validate -> 100%")
print(f"  Rollback: update-traffic PREVIOUS_REV=100 (instant)")


</details>


---
## Exercise 5 (Medium): OTel Monitoring


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

tools = [{"name":"search_courses","avg":120,"p99":450,"calls":5000,"err":0.02},
         {"name":"get_refund_policy","avg":45,"p99":180,"calls":800,"err":0.01},
         {"name":"calculate_emi","avg":15,"p99":60,"calls":1200,"err":0.005}]

print("OTel Monitoring:")
print(f"\n  Per-Tool Metrics:")
print(f"  {'Tool':<22} {'Avg':>6} {'P99':>6} {'Calls/d':>8} {'Err%':>6}")
for t in tools:
    print(f"  {t['name']:<22} {t['avg']:>4}ms {t['p99']:>4}ms {t['calls']:>8} {t['err']:>5.1%}")

print(f"\n  3 Essential Alerts:")
for sev, name, cond in [("CRITICAL","Error Rate","5xx > 5% for 5 min"),
    ("WARNING","Latency P99","> 2000ms for 5 min"),("WARNING","Instance Scale","instances > 8 of 10")]:
    print(f"    [{sev:<8}] {name}: {cond}")

print(f"\n  Structured Log:")
print(f"  {json.dumps({'severity':'INFO','tool':'search_courses','latency_ms':120,'success':True})}")

total = sum(t["calls"] for t in tools) * 30
budget = total * 0.005
burn = sum(t["calls"]*t["err"] for t in tools) * 30
print(f"\n  SLO Budget (99.5%): {budget:,.0f} allowed errors, {burn:,.0f} current, {budget-burn:,.0f} remaining")


</details>


---
## Exercise 6 (Challenge): OAuth 2.1 + PKCE
Design/architecture. See practice-lab-10_6.html.


In [ ]:
# Exercise 6: OAuth 2.1 + PKCE
pass


<details><summary>Solution</summary>


In [ ]:
print('OAuth 2.1 + PKCE: 8-step flow')
print('1. 401 -> 2. discover -> 3. AS metadata -> 4. register -> 5. authorize+PKCE -> 6. code -> 7. token -> 8. Bearer')
print('Token isolation: NEVER send server A token to server B (RFC 8707)')
print('IAM: GCP internal (zero-config). OAuth: external clients. Rate limit: Apigee/Redis')


</details>


---
## Exercise 7 (Challenge): Multi-Region Deploy
Design/architecture. See practice-lab-10_6.html.


In [ ]:
# Exercise 7: Multi-Region Deploy
pass


<details><summary>Solution</summary>


In [ ]:
print('Multi-Region: us-central1 + asia-south1 with Global HTTPS LB')
print('Latency: Mumbai 5-20ms vs Iowa 150-200ms from Hyderabad')
print('Cost: single Mumbai ~$8.50/mo. Dual + LB ~$22.25/mo')
print('Verdict: single Mumbai for India-only. Failover: RPO=0, RTO=30s')


</details>


---
## Exercise 8 (Challenge): India Bhashini on Cloud Run
Design/architecture. See practice-lab-10_6.html.


In [ ]:
# Exercise 8: India Bhashini on Cloud Run
pass


<details><summary>Solution</summary>


In [ ]:
print('Bhashini MCP on Cloud Run asia-south1')
print('4 tools x 22 languages. Cloud Run: ~$8-15/mo. E2E Networks: ~$5-15/hr')
print('DPDP: 72hr breach notif, 250Cr penalty, consent, erasure, audit logging')
print('Stack: MoSPI + Bhashini + Sarvam + Netsetos tools, all asia-south1')


</details>
